# Multi-task Reward Router Training (Utility + Task Type)

This notebook trains a multi-task VLM router that predicts:
1. **Task Type**: Classification head (which router task is this?)
2. **Utility-based Reward**: Regression head (predicting utility for `accuracy`, `cheap`, `fast`, `balanced` modes).

It loads data from a canonical parquet file, builds a multi-task dataset, trains the `MultiTaskRewardRouterModel`, and provides inference helpers.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import logging
import json
import random
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any, Union

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from transformers import AutoTokenizer, AutoModel, AutoConfig

# Set project root to allow importing modules if needed
# Assuming notebook is in artemis_final/router_train/notebooks
current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent
sys.path.append(str(PROJECT_ROOT))

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("multitask_router")

print(f"Project root set to: {PROJECT_ROOT}")
print(f"Torch version: {torch.__version__}")

In [ ]:
class RouterConfig:
    # Data
    data_path = "../../notebooks/data/router_profiles_with_utility.parquet"
    
    # Model
    text_encoder_name = "distilbert-base-uncased"
    max_seq_len = 256
    model_emb_dim = 32
    mode_emb_dim = 16
    hidden_dim = 256
    dropout = 0.1
    
    # Training
    batch_size = 256
    epochs = 10
    lr_encoder = 2e-5
    lr_head = 3e-4
    weight_decay = 0.01
    lambda_task = 0.3
    gradient_clip = 1.0
    seed = 42
    device = "cuda:1" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

cfg = RouterConfig()

# Set seeds
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)
print(f"Device: {cfg.device}")

In [14]:
def load_and_prep_data(path):
    p = Path(path)
    if not p.exists():
        # Try searching for it
        p_alt = Path("../data/router_profiles_with_utility.parquet")
        if p_alt.exists():
            p = p_alt
        else:
            raise FileNotFoundError(f"Could not find data at {path} or {p_alt}")
    
    print(f"Loading data from {p}...")
    df = pd.read_parquet(p)
    
    # Filter valid rows
    # Required cols
    req_cols = ["sample_id", "router_task", "data_split", "prompt_text", "model_name", "ok"]
    missing = [c for c in req_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    
    # Filter ok=True
    len_orig = len(df)
    df = df[df["ok"] == True].copy()
    print(f"Filtered ok=True: {len_orig} -> {len(df)}")
    
    return df

profiles_df = load_and_prep_data(cfg.data_path)
display(profiles_df.head(2))

Filtered ok=True: 339056 -> 339056


,sample_id,run_id,source_config,source_dataset,source_index,router_task,ground_truth_type,data_split,prompt_text,ground_truth,...,total_tokens,utility_accuracy,utility_cheap,utility_fast,utility_balanced,cost_norm_new,lat_norm,glider_score,judge_molmo_score,judge_molmo_rank_group
0,finqa_700_03a1c065,run_20251208_023438,finqa,cauldron,700,table_math,numeric,train,( 1 ) adjusted other income ( expense ) exclud...,\nRationale: the average free cash flow provid...,...,1442,0.484211,0.674716,0.686095,0.729454,0.039526,0.011078,0.0,9.0,2.0
1,docvqa_20_62702d0d,run_20251207_225050,docvqa,cauldron,20,document_ocr,exact,train,What is the name of the company?\nEnsure brevi...,B&W.,...,4449,1.000000,0.951137,0.994905,0.966276,0.122159,0.012737,5.0,10.0,1.0


In [15]:
MODES = ["accuracy", "cheap", "fast", "balanced"]

def create_long_format(df, modes):
    rows = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Building long-format"):
        base = {
            "sample_id": row["sample_id"],
            "router_task": row["router_task"],
            "data_split": row["data_split"],
            "prompt_text": row["prompt_text"],
            "model_name": row["model_name"],
            "prompt_len_words": row.get("txt_prompt_length_words", 0),
            "source_dataset": row.get("source_dataset", "unknown"),
        }
        
        # Add mode rows
        for mode in modes:
            # Check if utility_{mode} exists and is not null
            col_name = f"utility_{mode}"
            if col_name in row and pd.notnull(row[col_name]):
                r = base.copy()
                r["mode_name"] = mode
                r["utility_target"] = row[col_name]
                rows.append(r)
    
    return pd.DataFrame(rows)

df_long = create_long_format(profiles_df, MODES)
print(f"df_long shape: {df_long.shape}")
display(df_long.head())

Building long-format:   0%|          | 0/339056 [00:00<?, ?it/s]

df_long shape: (1356224, 9)


,sample_id,router_task,data_split,prompt_text,model_name,prompt_len_words,source_dataset,mode_name,utility_target
0,finqa_700_03a1c065,table_math,train,( 1 ) adjusted other income ( expense ) exclud...,qwen2_5_vl_7b,970,cauldron,accuracy,0.484211
1,finqa_700_03a1c065,table_math,train,( 1 ) adjusted other income ( expense ) exclud...,qwen2_5_vl_7b,970,cauldron,cheap,0.674716
2,finqa_700_03a1c065,table_math,train,( 1 ) adjusted other income ( expense ) exclud...,qwen2_5_vl_7b,970,cauldron,fast,0.686095
3,finqa_700_03a1c065,table_math,train,( 1 ) adjusted other income ( expense ) exclud...,qwen2_5_vl_7b,970,cauldron,balanced,0.729454
4,docvqa_20_62702d0d,document_ocr,train,What is the name of the company?\nEnsure brevi...,qwen2_5_vl_7b,12,cauldron,accuracy,1.000000


In [16]:
# Create indices
model_names = sorted(df_long["model_name"].unique())
mode_names = MODES
task_names = sorted(df_long["router_task"].unique())

model_to_id = {m: i for i, m in enumerate(model_names)}
mode_to_id = {m: i for i, m in enumerate(mode_names)}
task_to_id = {t: i for i, t in enumerate(task_names)}

# Map to IDs
df_long["model_id"] = df_long["model_name"].map(model_to_id)
df_long["mode_id"] = df_long["mode_name"].map(mode_to_id)
df_long["task_id"] = df_long["router_task"].map(task_to_id)

# Save indices
os.makedirs("data", exist_ok=True)
with open("data/model_index.json", "w") as f: json.dump(model_names, f)
with open("data/mode_index.json", "w") as f: json.dump(mode_names, f)
with open("data/task_index.json", "w") as f: json.dump(task_names, f)

print("Indices saved to data/")
print(f"Models: {len(model_names)}")
print(f"Modes: {len(mode_names)}")
print(f"Tasks: {len(task_names)}")

Indices saved to data/
Models: 5
Modes: 4
Tasks: 30


In [17]:
class MultiTaskRewardRouterDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Build prompt text
        # [ROUTER] Task: {router_task}. Source: {source_dataset}. Split: {data_split}. PromptLen: {len}. Question: {text}
        input_text = (
            f"[ROUTER] Task: {row['router_task']}. "
            f"Source: {row['source_dataset']}. "
            f"PromptLen: {row['prompt_len_words']}. "
            f"Question: {row['prompt_text']}"
        )
        
        encoding = self.tokenizer(
            input_text,
            max_length=self.max_length,
            truncation=True,
            padding=False, # We handle padding in collate
            return_tensors=None
        )
        
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "model_id": row["model_id"],
            "mode_id": row["mode_id"],
            "utility_target": float(row["utility_target"]),
            "task_id": row["task_id"],
            "sample_id": row["sample_id"]
        }

def multitask_collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    attention_mask = [item["attention_mask"] for item in batch]
    
    # Pad sequences
    max_len = max(len(x) for x in input_ids)
    input_ids_padded = []
    attention_mask_padded = []
    
    for ids, mask in zip(input_ids, attention_mask):
        pad_len = max_len - len(ids)
        input_ids_padded.append(ids + [0] * pad_len)
        attention_mask_padded.append(mask + [0] * pad_len)
        
    return {
        "input_ids": torch.tensor(input_ids_padded, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask_padded, dtype=torch.long),
        "model_id": torch.tensor([x["model_id"] for x in batch], dtype=torch.long),
        "mode_id": torch.tensor([x["mode_id"] for x in batch], dtype=torch.long),
        "utility_target": torch.tensor([x["utility_target"] for x in batch], dtype=torch.float),
        "task_id": torch.tensor([x["task_id"] for x in batch], dtype=torch.long),
        "sample_ids": [x["sample_id"] for x in batch]
    }

In [18]:
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(cfg.text_encoder_name)

# Split data
train_df = df_long[df_long["data_split"] == "train"]
val_df = df_long[df_long["data_split"] == "val"]
test_df = df_long[df_long["data_split"] == "test"]

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

train_ds = MultiTaskRewardRouterDataset(train_df, tokenizer, cfg.max_seq_len)
val_ds = MultiTaskRewardRouterDataset(val_df, tokenizer, cfg.max_seq_len)
test_ds = MultiTaskRewardRouterDataset(test_df, tokenizer, cfg.max_seq_len)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=multitask_collate_fn)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=multitask_collate_fn)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=multitask_collate_fn)

# Verify batch
batch = next(iter(train_loader))
print("Batch shapes:")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"{k}: {v.shape}")

Train: 951912, Val: 204452, Test: 199860
Batch shapes:
input_ids: torch.Size([256, 256])
attention_mask: torch.Size([256, 256])
model_id: torch.Size([256])
mode_id: torch.Size([256])
utility_target: torch.Size([256])
task_id: torch.Size([256])


In [19]:
class MultiTaskRewardRouterModel(nn.Module):
    def __init__(self, config, num_models, num_modes, num_tasks):
        super().__init__()
        self.config = config
        
        # Text Encoder
        self.text_encoder = AutoModel.from_pretrained(config.text_encoder_name)
        text_hidden_size = self.text_encoder.config.hidden_size
        
        # Embeddings
        self.model_embedding = nn.Embedding(num_models, config.model_emb_dim)
        self.mode_embedding = nn.Embedding(num_modes, config.mode_emb_dim)
        
        # Routing Head
        # Input: [text, model, mode]
        input_dim = text_hidden_size + config.model_emb_dim + config.mode_emb_dim
        self.routing_mlp = nn.Sequential(
            nn.Linear(input_dim, config.hidden_dim),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(config.hidden_dim // 2, 1) # scalar utility
        )
        
        # Task Head
        # Input: [text] only
        self.task_head = nn.Sequential(
            nn.Linear(text_hidden_size, config.hidden_dim),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, num_tasks)
        )
        
    def forward(self, input_ids, attention_mask, model_id, mode_id):
        # Encode text
        outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Use CLS token (index 0)
        h_text = outputs.last_hidden_state[:, 0, :]
        
        # Task Prediction (Text only)
        task_logits = self.task_head(h_text)
        
        # Utility Prediction
        h_model = self.model_embedding(model_id)
        h_mode = self.mode_embedding(mode_id)
        
        h_combined = torch.cat([h_text, h_model, h_mode], dim=-1)
        utility_hat = self.routing_mlp(h_combined).squeeze(-1)
        
        return {
            "utility_hat": utility_hat,
            "task_logits": task_logits
        }

model = MultiTaskRewardRouterModel(
    cfg, 
    num_models=len(model_names),
    num_modes=len(mode_names),
    num_tasks=len(task_names)
)
model.to(cfg.device)
print("Model initialized.")

Model initialized.


In [20]:
optimizer = torch.optim.AdamW([
    {"params": model.text_encoder.parameters(), "lr": cfg.lr_encoder},
    {"params": model.routing_mlp.parameters(), "lr": cfg.lr_head},
    {"params": model.task_head.parameters(), "lr": cfg.lr_head},
    {"params": model.model_embedding.parameters(), "lr": cfg.lr_head},
    {"params": model.mode_embedding.parameters(), "lr": cfg.lr_head},
], weight_decay=cfg.weight_decay)

num_training_steps = len(train_loader) * cfg.epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_training_steps)

criterion_utility = nn.MSELoss()
criterion_task = nn.CrossEntropyLoss()

best_val_loss = float("inf")
trigger_times = 0
patience = 3

history = []

for epoch in range(cfg.epochs):
    # Train
    model.train()
    train_loss = 0
    train_loss_util = 0
    train_loss_task = 0
    
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs} [Train]"):
        batch = {k: v.to(cfg.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        
        optimizer.zero_grad()
        
        out = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            model_id=batch["model_id"],
            mode_id=batch["mode_id"]
        )
        
        loss_u = criterion_utility(out["utility_hat"], batch["utility_target"])
        loss_t = criterion_task(out["task_logits"], batch["task_id"])
        
        loss = loss_u + cfg.lambda_task * loss_t
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip)
        optimizer.step()
        scheduler.step()
        
        train_loss += loss.item()
        train_loss_util += loss_u.item()
        train_loss_task += loss_t.item()
        
    # Val
    model.eval()
    val_loss = 0
    val_loss_util = 0
    val_loss_task = 0
    val_task_corr = 0
    val_total = 0
    
    val_preds_u = []
    val_targets_u = []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            batch = {k: v.to(cfg.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            
            out = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                model_id=batch["model_id"],
                mode_id=batch["mode_id"]
            )
            
            loss_u = criterion_utility(out["utility_hat"], batch["utility_target"])
            loss_t = criterion_task(out["task_logits"], batch["task_id"])
            loss = loss_u + cfg.lambda_task * loss_t
            
            val_loss += loss.item()
            val_loss_util += loss_u.item()
            val_loss_task += loss_t.item()
            
            # Task Acc
            preds_t = torch.argmax(out["task_logits"], dim=1)
            val_task_corr += (preds_t == batch["task_id"]).sum().item()
            val_total += len(batch["task_id"])
            
            val_preds_u.extend(out["utility_hat"].cpu().numpy())
            val_targets_u.extend(batch["utility_target"].cpu().numpy())
            
    # Metrics
    train_stats = {
        "loss": train_loss / len(train_loader),
        "u_loss": train_loss_util / len(train_loader),
        "t_loss": train_loss_task / len(train_loader)
    }
    
    val_stats = {
        "loss": val_loss / len(val_loader),
        "u_loss": val_loss_util / len(val_loader),
        "t_loss": val_loss_task / len(val_loader),
        "t_acc": val_task_corr / val_total
    }
    
    try:
        pearson = pearsonr(val_preds_u, val_targets_u)[0]
    except: pearson = 0
    
    print(f"Epoch {epoch+1}: Train Loss {train_stats['loss']:.4f} | Val Loss {val_stats['loss']:.4f} | Val Task Acc {val_stats['t_acc']:.4f} | Val Pearson {pearson:.4f}")
    
    # Early Stopping
    if val_stats["loss"] < best_val_loss:
        best_val_loss = val_stats["loss"]
        torch.save(model.state_dict(), "best_multitask_router.pt")
        trigger_times = 0
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print("Early stopping!")
            break

Epoch 1/10 [Train]:   0%|          | 0/3719 [00:00<?, ?it/s]

Epoch 1 [Val]:   0%|          | 0/799 [00:00<?, ?it/s]

Epoch 1: Train Loss 0.0724 | Val Loss 0.0608 | Val Task Acc 1.0000 | Val Pearson 0.6916


Epoch 2/10 [Train]:   0%|          | 0/3719 [00:00<?, ?it/s]

Epoch 2 [Val]:   0%|          | 0/799 [00:00<?, ?it/s]

...

Epoch 3 [Val]:   0%|          | 0/799 [00:00<?, ?it/s]

Epoch 3: Train Loss 0.0440 | Val Loss 0.0654 | Val Task Acc 1.0000 | Val Pearson 0.6773


Epoch 4/10 [Train]:   0%|          | 0/3719 [00:00<?, ?it/s]

Epoch 4 [Val]:   0%|          | 0/799 [00:00<?, ?it/s]

Epoch 4: Train Loss 0.0377 | Val Loss 0.0669 | Val Task Acc 1.0000 | Val Pearson 0.6676
Early stopping!


In [23]:
# Load Best Model
model.load_state_dict(torch.load("best_multitask_router.pt", map_location=cfg.device, weights_only=False))
model.eval()
print("Loaded best model.")

# Evaluate on Test
test_task_corr = 0
test_total = 0
test_preds_u = []
test_targets_u = []
test_modes = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test Eval"):
        batch = {k: v.to(cfg.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        out = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            model_id=batch["model_id"],
            mode_id=batch["mode_id"]
        )
        
        preds_t = torch.argmax(out["task_logits"], dim=1)
        test_task_corr += (preds_t == batch["task_id"]).sum().item()
        test_total += len(batch["task_id"])
        
        test_preds_u.extend(out["utility_hat"].cpu().numpy())
        test_targets_u.extend(batch["utility_target"].cpu().numpy())
        test_modes.extend(batch["mode_id"].cpu().numpy())

print(f"Test Task Accuracy: {test_task_corr / test_total:.4f}")
print(f"Test Utility Pearson: {pearsonr(test_preds_u, test_targets_u)[0]:.4f}")

# Per Mode Pearson
df_res = pd.DataFrame({"pred": test_preds_u, "target": test_targets_u, "mode_id": test_modes})
df_res["mode"] = df_res["mode_id"].map({v: k for k, v in mode_to_id.items()})
for mode in mode_names:
    sub = df_res[df_res["mode"] == mode]
    if len(sub) > 1:
        p = pearsonr(sub["pred"], sub["target"])[0]
        print(f"Pearson [{mode}]: {p:.4f}")

Loaded best model.


Test Eval:   0%|          | 0/781 [00:00<?, ?it/s]

Test Task Accuracy: 1.0000
Test Utility Pearson: 0.6858
Pearson [accuracy]: 0.6578
Pearson [cheap]: 0.6482
Pearson [fast]: 0.6512
Pearson [balanced]: 0.6479


In [24]:
print("Evaluating Routing Accuracy vs Oracle...")
# We need to group by sample_id and mode, and compare predicted best vs oracle best

# 1. Get predictions for ALL test rows
test_df["pred_utility"] = test_preds_u

results = []
samples = test_df["sample_id"].unique()

# It's faster to do this using pandas groupby
for mode in mode_names:
    mode_df = test_df[test_df["mode_name"] == mode]
    
    total_samples = 0
    routing_hits = 0
    oracle_utility_sum = 0
    router_utility_sum = 0
    
    # Group by sample
    for sid, group in mode_df.groupby("sample_id"):
        if len(group) < 2: continue
        
        # Oracle
        oracle_idx = group["utility_target"].idxmax()
        oracle_row = group.loc[oracle_idx]
        
        # Predicted
        pred_idx = group["pred_utility"].idxmax()
        pred_row = group.loc[pred_idx]
        
        total_samples += 1
        if pred_row["model_name"] == oracle_row["model_name"]:
            routing_hits += 1
            
        oracle_utility_sum += oracle_row["utility_target"]
        router_utility_sum += pred_row["utility_target"]
        
    if total_samples > 0:
        acc = routing_hits / total_samples
        oracle_mean = oracle_utility_sum / total_samples
        router_mean = router_utility_sum / total_samples
        gap = oracle_mean - router_mean
        recovery = router_mean / oracle_mean if oracle_mean > 0 else 0
        
        results.append({
            "mode": mode,
            "routing_acc": acc,
            "oracle_util": oracle_mean,
            "router_util": router_mean,
            "gap": gap,
            "recovery": recovery
        })

res_df = pd.DataFrame(results)
display(res_df)

Evaluating Routing Accuracy vs Oracle...


/home/hice1/vchopra37/scratch/models/tmp/ipykernel_289051/1043658028.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["pred_utility"] = test_preds_u


,mode,routing_acc,oracle_util,router_util,gap,recovery
0,accuracy,0.359251,0.771734,0.617835,0.153899,0.800580
1,cheap,0.292605,0.844086,0.690781,0.153305,0.818377
2,fast,0.306915,0.843565,0.697316,0.146249,0.826630
3,balanced,0.351946,0.864689,0.780489,0.084199,0.902625


In [29]:
def route_query(
    model,
    tokenizer,
    prompt_text: str,
    mode_name: str,
    model_names: List[str],
    model_to_id: Dict[str, int],
    mode_to_id: Dict[str, int],
    id_to_task: List[str],     # <-- FIX: must be id_to_task, NOT task_index list
    device: str = "cuda",
    temperature: float = 1.0,
    task_top_k: int = 3,       # <-- NEW: configurable top-k tasks
):
    """
    Given a single prompt and mode, compute:
      - model utility scores + probabilities
      - top-k task predictions (names + probs)
      - predicted task_type and confidence
    """

    model.eval()

    # Build inference input text
    input_text = (
        f"[ROUTER] Task: unknown. "
        f"Source: inference. "
        f"PromptLen: {len(prompt_text.split())}. "
        f"Question: {prompt_text}"
    )

    encoding = tokenizer(
        input_text,
        max_length=256,
        truncation=True,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    # Repeat inputs for each model
    num_cands = len(model_names)
    input_ids = input_ids.repeat(num_cands, 1)
    attention_mask = attention_mask.repeat(num_cands, 1)

    # Model & mode IDs
    model_ids = torch.tensor([model_to_id[m] for m in model_names], device=device)
    mode_id_val = mode_to_id.get(mode_name, 0)
    mode_ids = torch.tensor([mode_id_val] * num_cands, device=device)

    # Forward pass
    with torch.no_grad():
        out = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            model_id=model_ids,
            mode_id=mode_ids
        )

        utility_scores = out["utility_hat"]          # shape [num_models]
        task_logits    = out["task_logits"][0]       # we only need first row

    # ------------------------------------------------
    #   MODEL ROUTING PROBABILITIES
    # ------------------------------------------------
    scores = utility_scores / temperature
    probs  = torch.softmax(scores, dim=0)

    scores_dict = {m: float(s) for m, s in zip(model_names, utility_scores)}
    probs_dict  = {m: float(p) for m, p in zip(model_names, probs)}

    # ------------------------------------------------
    #   TASK CLASSIFICATION + TOP-K
    # ------------------------------------------------
    task_probs = torch.softmax(task_logits, dim=0)

    # Top-1 task prediction
    task_pred_idx  = torch.argmax(task_probs).item()
    task_pred_name = id_to_task[task_pred_idx]
    task_conf      = float(task_probs[task_pred_idx])

    # Top-k tasks
    topk_probs, topk_indices = torch.topk(task_probs, k=min(task_top_k, len(task_probs)))

    task_topk = []
    for p, idx in zip(topk_probs.tolist(), topk_indices.tolist()):
        task_topk.append({
            "task": id_to_task[idx],
            "prob": float(p),
        })

    # Full task prob dictionary
    task_probs_dict = {
        id_to_task[i]: float(task_probs[i])
        for i in range(len(task_probs))
    }

    # ------------------------------------------------
    #   OUTPUT
    # ------------------------------------------------
    return {
        "scores": scores_dict,
        "probs": probs_dict,
        "task_probs": task_probs_dict,
        "task_type_pred": task_pred_name,
        "task_confidence": task_conf,
        "task_topk": task_topk,              # <-- NEW: top-K tasks
    }


In [31]:
# Demo Inference
sample_text = "Analyze this image and tell me if the chart shows positive growth."
candidates = model_names[:3]
mode = "accuracy"

result = route_query(
    model,
    tokenizer,
    sample_text,
    mode,
    candidates,
    model_to_id,
    mode_to_id,
    task_names,
    device=cfg.device
)


print(f"Prompt: {sample_text}")
print(f"Predicted Task: {result['task_type_pred']} ({result['task_confidence']:.2%})")
print("Top-K Task Predictions:")
for t in result["task_topk"]:
    print(f"  {t['task']}: {t['prob']:.2%}")
print("\nRouting:")
for m in candidates:
    print(f"{m}: Score={result['scores'][m]:.3f}, Prob={result['probs'][m]:.2%}")

Prompt: Analyze this image and tell me if the chart shows positive growth.
Predicted Task: difference_detection (42.30%)
Top-K Task Predictions:
  difference_detection: 42.30%
  map_reasoning: 38.02%
  chart_captioning: 6.80%

Routing:
deepseek_ocr: Score=0.020, Prob=26.94%
gemma_3_27b: Score=0.371, Prob=38.26%
qwen2_5_vl_3b: Score=0.276, Prob=34.80%
